In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import numpy as np
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from sorl.sorl_wrapper import SorlModelWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model + checkpoint
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(model_name, abstract_vocab_size_list=[128])

hf_repo_id = "Ksgk-fy/sorl_pt"
hf_filename = "qwen2.5-0.5B_gsm8k_K4_v128_i2/final.pt"
ckpt_path = hf_hub_download(repo_id=hf_repo_id, filename=hf_filename)
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt["model"] if "model" in ckpt else ckpt
clean_sd = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(clean_sd)
model = model.to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)
K = 4

print(f"Device: {device} | Step: {ckpt.get('step', 'N/A')} | Epoch: {ckpt.get('epoch', 'N/A')}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Device: cpu | Step: 2805 | Epoch: 3


In [ ]:
# ── Demo: SoRLCompressTrainer with inner-cot mode (1 training step) ──────
from sorl.trainer_compress import SoRLCompressTrainer, SoRLCompressConfig
from data.pt_dataset import get_dataset, collate_fn

train_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)

cfg = SoRLCompressConfig(
    inner_cot=True,
    n_inner_cot_tokens=8,
    alpha_distill=1.0,
    distill_temperature=2.0,
    num_rollouts=2,
    K=K,
    max_iterations=2,
    batch_size=2,
    lr=1e-5,
)

trainer = SoRLCompressTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    collate_fn=collate_fn,
    config=cfg,
    device=str(device),
)

# Run a single training step
model.train()
dl = trainer._make_dataloader(train_ds, shuffle=True)
batch = next(iter(dl))
loss_dict = trainer._training_step(batch)

print("── Single training step (inner-cot) ──")
for k, v in loss_dict.items():
    print(f"  {k:>20s}: {v.item():.4f}")
model.eval()
print("Done ✓")